# Plant Leaf Diseases Classification CNN (TensorFlow Implementation)
This notebook loads the converted PyTorch-trained MobileNetV2 model and provides optional retraining in TensorFlow.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)

AUTOTUNE = tf.data.AUTOTUNE
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
DATASET_DIR = '../DATASET/Augmentated/Plant_leave_diseases_dataset_with_augmentation'
NUM_CLASSES = 39

## 1. Load Dataset with tf.data

In [ ]:
train_ds = keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='training',
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

val_ds = keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='validation',
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

class_names = train_ds.class_names
print(f"Total classes: {len(class_names)}")
print(f"Training batches: {len(train_ds)}")
print(f"Validation batches: {len(val_ds)}")

## 2. Data Augmentation & Prefetching

In [ ]:
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.15),
    keras.layers.RandomContrast(0.2),
])

def preprocess(image, label):
    image = tf.cast(image, tf.float32)
    return image, label

train_ds = train_ds.map(preprocess, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)

val_ds = val_ds.map(preprocess, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 3. Class Weights for Imbalance

In [ ]:
all_labels = np.concatenate([y.numpy() for _, y in train_ds], axis=0)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(all_labels),
    y=all_labels
)
class_weight_dict = {i: class_weights[i] for i in range(NUM_CLASSES)}
print('Class weights computed.')

## 4. Build & Load Converted Model

In [ ]:
model_path = 'cnn_mobilenet_tf.keras'
if os.path.exists(model_path):
    model = keras.models.load_model(model_path)
    print(f"Loaded converted TF model from {model_path}")
else:
    print(f"{model_path} not found. Build from scratch? (run convert_weights.py first)")

## 5. Optional: Retrain Classifier Head

In [ ]:
RETRAIN = True  # Set to True to fine-tune

if RETRAIN:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
        metrics=['accuracy']
    )

    callbacks = [
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint('cnn_mobilenet_tf_best.keras', save_best_only=True, monitor='val_loss')
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=30,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1
    )
    
    model.save('cnn_mobilenet_tf_final.keras')
    print('Retrained model saved.')

## 6. Evaluation

In [ ]:
def evaluate_model(model, val_ds, class_names):
    all_preds, all_labels = [], []
    for images, labels in val_ds:
        preds = tf.argmax(model.predict(images, verbose=0), axis=1)
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

    print("\n--- Classification Report ---")
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(24, 20))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted', fontsize=16)
    plt.ylabel('True', fontsize=16)
    plt.title('Confusion Matrix', fontsize=20)
    plt.xticks(rotation=90, fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    plt.tight_layout()
    plt.show()

evaluate_model(model, val_ds, class_names)